# 13 — เตรียมข้อมูลระดับตำบล จังหวัดสระบุรี (สำหรับ LISA และ SaTScan)

Notebook นี้ **ดึงข้อมูลจริงทั้งหมดจากอินเทอร์เน็ต** แล้วรวมเป็นชุดข้อมูลรายตำบลชุดเดียว พร้อม **ตรวจคุณภาพ (QC)** ก่อนนำไปวิเคราะห์ใน Notebook 15 (LISA) และ 16 (SaTScan)

| ข้อมูล | แหล่ง | ผลลัพธ์ |
|---|---|---|
| ขอบเขตตำบล | OCHA COD-AB Thailand (รหัสตำบลตรงกับ HDC) → สำรอง GADM 4.1 → OpenStreetMap | polygon + ชื่อ + พื้นที่ (km²) |
| ประชากร | WorldPop 2020 (zonal sum) | ประชากรรายตำบล |
| จุดความร้อน | NASA FIRMS (VIIRS 375 m) | จำนวนรายปี / รายเดือน |
| (ทางเลือก) ผู้ป่วย | ไฟล์ HDC ของคุณใน Drive | จำนวนผู้ป่วยรายตำบล + รายงานการจับคู่ชื่อ |

> Notebook 15 และ 16 **รันได้เองโดยไม่ต้องใช้ A** (มีขั้นตอนเตรียมข้อมูลชุดเดียวกันอยู่แล้ว) — A มีไว้เพื่อ **ตรวจข้อมูลให้ละเอียดก่อน** และบันทึกชุดข้อมูลกลางไว้ใน Drive (`SAVE_TO_DRIVE=True`)

## 0. CONFIG

In [ ]:
# ===== Repository data (data/processed/) =====
# Opened from GitHub in Colab: the repository is cloned so that data/processed/ is available.
# Run locally: start Jupyter in the repository (or its notebooks/ folder).
# If data/processed/ is not found, the notebook falls back to the original Google Drive paths.
import os, subprocess
REPO_URL = "https://github.com/songwutsa/paper3-pm25-acute-illness-saraburi"
USE_REPO = True          # False = use the original Google Drive paths only
REPO_DIR = None
for _d in (os.getcwd(), os.path.dirname(os.getcwd()), "/content/paper3-pm25-acute-illness-saraburi"):
    if os.path.isdir(os.path.join(_d, "data", "processed")):
        REPO_DIR = _d; break
if USE_REPO and REPO_DIR is None and os.path.isdir("/content"):
    REPO_DIR = "/content/paper3-pm25-acute-illness-saraburi"
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR], check=False)
REPO_DATA = os.path.join(REPO_DIR, "data", "processed") if REPO_DIR else ""
USE_REPO_DATA = bool(USE_REPO and REPO_DATA and os.path.isdir(REPO_DATA))

def repo_file(name):
    """Path of a file in data/processed/ (matched by file name), or None."""
    p = os.path.join(REPO_DATA, os.path.basename(str(name))) if USE_REPO_DATA else ""
    return p if p and os.path.exists(p) else None

print("Repository data:", REPO_DATA if USE_REPO_DATA else "not found - using Google Drive paths")

In [ ]:
PROVINCE_EN = "Saraburi"
YEARS = list(range(2016, 2027))   # จุดความร้อน FIRMS 2016–2026 (ปีที่ไม่มีข้อมูลจะถูกตัดออกพร้อมแจ้ง)
BOUNDARY_SOURCE = "drive"       # "drive" (shapefile ตำบลใน Google Drive) → สำรอง "cod" → "gadm" → "osm" อัตโนมัติ
TAMBON_SHP = ""                 # path ไฟล์ขอบเขตตำบลใน Drive เช่น "GIS/tambon_saraburi.shp" (.shp/.gpkg/.zip/.geojson) — เว้นว่าง = ค้นหาอัตโนมัติ
FIRMS_MAP_KEY = ""                # free key: https://firms.modaps.eosdis.nasa.gov/api/map_key (or Colab secret FIRMS_MAP_KEY)
EXPECTED_TAMBONS = 108              # จำนวนตำบลตามข้อมูล HDC — ใช้ตรวจสอบ
SAVE_TO_DRIVE = False

In [ ]:
import importlib, importlib.util, subprocess, sys
need = {"esda": "esda", "libpysal": "libpysal", "rasterio": "rasterio", "mapclassify": "mapclassify", "osmnx": "osmnx"}
missing = [p for m, p in need.items() if importlib.util.find_spec(m) is None]
if missing:
    print("ติดตั้ง:", missing); subprocess.run(f"{sys.executable} -m pip -q install " + " ".join(missing), shell=True)

In [ ]:
import os, io, json, math, time, zipfile, warnings, subprocess
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

UA = {"User-Agent": "Mozilla/5.0 (academic research notebook; PM2.5 Saraburi)"}

def setup_thai_font():
    """ให้กราฟ matplotlib แสดงภาษาไทยได้ (ฟอนต์เริ่มต้นไม่มีตัวอักษรไทย → แสดงเป็น □□□)"""
    import matplotlib.font_manager as fm, glob
    base = "/content" if os.path.isdir("/content") else "."
    path = os.path.join(base, "Sarabun-Regular.ttf")
    if not os.path.exists(path):
        try:
            r = requests.get("https://github.com/google/fonts/raw/main/ofl/sarabun/Sarabun-Regular.ttf", timeout=60)
            if r.status_code == 200 and len(r.content) > 50000: open(path, "wb").write(r.content)
        except Exception:
            pass
    if not os.path.exists(path):                                  # สำรอง: ฟอนต์ไทยจาก apt (TLWG)
        subprocess.run("apt-get -qq install -y fonts-thai-tlwg > /dev/null 2>&1", shell=True)
        cand = sorted(glob.glob("/usr/share/fonts/**/tlwg/Loma.*tf", recursive=True)) + \
               sorted(glob.glob("/usr/share/fonts/**/Garuda.*tf", recursive=True))
        path = cand[0] if cand else None
    if path and os.path.exists(path):
        fm.fontManager.addfont(path)
        name = fm.FontProperties(fname=path).get_name()
        plt.rcParams["font.family"] = ["DejaVu Sans", name]      # ละติน = DejaVu, ไทย = fallback ไปฟอนต์ไทย
        print(f"✓ ฟอนต์ไทยสำหรับกราฟ: {name}")
    else:
        print("⚠ ติดตั้งฟอนต์ไทยไม่ได้ — ข้อความภาษาไทยในกราฟจะแสดงเป็นสี่เหลี่ยม")
setup_thai_font()

# เว็บราชการบางแห่ง (เช่น Air4Thai) ส่งใบรับรอง SSL ไม่ครบ chain → Python ตรวจสอบไม่ผ่าน (SSLError)
# อนุญาตให้ข้ามการตรวจใบรับรองเฉพาะโดเมนเหล่านี้ (ข้อมูลสาธารณะ อ่านอย่างเดียว)
SSL_RELAX_HOSTS = ("air4thai.pcd.go.th", "air4thai.com", "www.air4thai.com")

def http_get(url, timeout=(15, 120), tries=3, quiet=False, **kw):
    """GET พร้อม retry — คืน Response เมื่อ HTTP 200 ไม่เช่นนั้น raise"""
    last = None
    host = url.split("/")[2] if "://" in url else ""
    verify = kw.pop("verify", True)
    for k in range(tries):
        try:
            try:
                r = requests.get(url, headers=UA, timeout=timeout, verify=verify, **kw)
            except requests.exceptions.SSLError:
                if host not in SSL_RELAX_HOSTS or verify is False:
                    raise
                import urllib3; urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
                if not quiet: print(f"   (ใบรับรอง SSL ของ {host} ไม่สมบูรณ์ — เชื่อมต่อโดยไม่ตรวจใบรับรอง)")
                verify = False
                r = requests.get(url, headers=UA, timeout=timeout, verify=False, **kw)
            if r.status_code == 200:
                return r
            last = f"HTTP {r.status_code}"
            if r.status_code == 429 and k < tries - 1:     # ขอถี่เกินโควตา (เช่น Open-Meteo) → รอนานขึ้นแล้วลองใหม่
                wait = int(r.headers.get("Retry-After", 0) or 0) or 30 * (k + 1)
                print(f"   HTTP 429 จาก {host} — รอ {wait} s แล้วลองใหม่", flush=True)
                time.sleep(wait); continue
        except Exception as e:
            last = type(e).__name__ + ": " + str(e)[:100]
        if k < tries - 1:
            time.sleep(2 * (k + 1))
    raise RuntimeError(f"{last}  <- {url[:110]}")

def month_chunks(start, end):
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    out, cur = [], s
    while cur <= e:
        nxt = min(cur + pd.offsets.MonthEnd(0), e)
        out.append((cur.strftime("%Y-%m-%d"), nxt.strftime("%Y-%m-%d")))
        cur = nxt + pd.Timedelta(days=1)
    return out

In [ ]:
# ---- ที่เก็บผลลัพธ์ (ถ้า SAVE_TO_DRIVE=True จะเก็บใน Google Drive เพื่อให้ Notebook อื่นใช้ต่อได้) ----
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUTDIR = "/content/drive/MyDrive/PM25_Saraburi_notebooks"
    except Exception as e:
        print("⚠ mount Drive ไม่ได้ จะเก็บไว้ใน /content แทน:", e)
        OUTDIR = "/content/PM25_Saraburi_notebooks"
else:
    OUTDIR = "/content/PM25_Saraburi_notebooks" if os.path.isdir("/content") else "./PM25_Saraburi_notebooks"
os.makedirs(OUTDIR, exist_ok=True)
print("OUTDIR =", OUTDIR)

## 1. ดึงข้อมูลจริง

In [ ]:
# =====================================================================
#  เตรียมข้อมูลระดับตำบล (ดึงจากอินเทอร์เน็ตทั้งหมด)
#  1) ขอบเขตตำบล  : shapefile ใน Google Drive → OCHA COD-AB Thailand (มีรหัสตำบล 6 หลักตรงกับ HDC) → สำรอง GADM 4.1 → OpenStreetMap
#  2) ประชากร      : WorldPop 2020 (1 km, ปรับรวมรายตำบลด้วย zonal sum)
#  3) จุดความร้อน  : NASA FIRMS (country yearly summary; สำรอง: API + MAP_KEY)
# =====================================================================
import geopandas as gpd
from shapely.geometry import Point
UTM = "EPSG:32647"

COD_BASE = "https://codgis.itos.uga.edu/arcgis/rest/services/COD_External/THA_pcode/FeatureServer"
GADM_URL = "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_THA_3.json.zip"

def load_tambons_cod():
    """OCHA COD-AB Thailand (ที่มา: กรมแผนที่ทหาร/RTSD) ผ่าน live service ของ ITOS — มีรหัสตำบล (pcode) และชื่อภาษาไทย"""
    import re
    from urllib.parse import quote
    meta = http_get(COD_BASE + "?f=json", timeout=(15, 60)).json()
    layers = meta.get("layers", [])
    lyr = next((l for l in layers if re.search(r"(adm(in)?\s*_?3|tambon|sub.?district)", l["name"], re.I)), None)
    if lyr is None:
        raise RuntimeError(f"ไม่พบชั้นข้อมูลระดับ 3: {[l['name'] for l in layers]}")
    for where in [f"ADM1_EN = '{PROVINCE_EN}'", "ADM1_PCODE = 'TH19'", f"ADM1_EN LIKE '%{PROVINCE_EN}%'"]:
        url = f"{COD_BASE}/{lyr['id']}/query?where={quote(where)}&outFields=*&outSR=4326&f=geojson"
        try:
            g = gpd.read_file(io.BytesIO(http_get(url, timeout=(15, 300), tries=2).content))
        except Exception:
            continue
        if len(g):
            break
    else:
        raise RuntimeError("query ไม่ได้ข้อมูล")
    g = g.set_crs("EPSG:4326", allow_override=True)
    namecol = next(c for c in ["ADM3_TH", "ADM3_EN", "admin3Name_th", "admin3Name_en"] if c in g)
    pc = next(c for c in ["ADM3_PCODE", "admin3Pcode"] if c in g)
    g["tambon_id"] = g[pc]; g["name"] = g[namecol]
    g["a_code"] = g[pc].astype(str).str.replace(r"\D", "", regex=True).str[:6]     # TH190101 → 190101 (ตรงกับ HDC)
    if "ADM3_EN" in g: g["name_en"] = g["ADM3_EN"]
    return g[["tambon_id", "a_code", "name"] + (["name_en"] if "name_en" in g else []) + ["geometry"]], \
           f"OCHA COD-AB Thailand (ITOS live service, layer '{lyr['name']}')"

def load_tambons_gadm():
    """GADM 4.1 ระดับ 3 (ตำบล) — ไฟล์ทั้งประเทศ ~30–60 MB"""
    print("   ดาวน์โหลด GADM 4.1 THA level 3 ...")
    raw = http_get(GADM_URL, timeout=(15, 900)).content
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        name = next(n for n in z.namelist() if n.endswith(".json"))
        g = gpd.read_file(io.BytesIO(z.read(name)))
    g = g[g["NAME_1"].str.replace(" ", "").str.contains(PROVINCE_EN, case=False, na=False)].copy()
    g["tambon_id"] = g["GID_3"]
    g["name"] = g["NL_NAME_3"].where(g.get("NL_NAME_3", pd.Series(dtype=str)).notna() & (g.get("NL_NAME_3", "") != "NA"), g["NAME_3"]) \
        if "NL_NAME_3" in g else g["NAME_3"]
    g["name_en"] = g["NAME_3"]
    cols = ["tambon_id", "name", "name_en", "geometry"]
    if "CC_3" in g and g["CC_3"].astype(str).str.fullmatch(r"\d{6}").mean() > 0.8:
        g["a_code"] = g["CC_3"].astype(str); cols.insert(1, "a_code")
    return g[cols].to_crs("EPSG:4326"), "GADM 4.1 (THA level 3)"

def load_tambons_osm():
    import osmnx as ox
    ox.settings.timeout = 600
    prov = ox.geocode_to_gdf(f"{PROVINCE_EN} Province, Thailand")
    # หมายเหตุ: osmnx รวมแท็กแบบ OR → ถ้าใส่ทั้ง boundary และ admin_level จะได้ขอบเขตทุกระดับ (จังหวัด/อำเภอ/ตำบล) ปนกัน
    g = ox.features_from_polygon(prov.geometry.iloc[0], {"admin_level": "8"}).reset_index()
    g = g[g.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()
    if "boundary" in g: g = g[g["boundary"].astype(str) == "administrative"]
    if "admin_level" in g: g = g[g["admin_level"].astype(str) == "8"]
    if "element" in g and (g["element"] == "relation").any(): g = g[g["element"] == "relation"]
    g = g[g.representative_point().within(prov.geometry.iloc[0])]
    namecol = next(c for c in ["name:th", "name", "name:en"] if c in g)
    g["tambon_id"] = g.get("id", g.index).astype(str); g["name"] = g[namecol]
    if "name:en" in g: g["name_en"] = g["name:en"]
    return g[["tambon_id", "name", "geometry"] + (["name_en"] if "name_en" in g else [])], "OpenStreetMap admin_level=8"

# ---------- ขอบเขตตำบลจาก shapefile ใน Google Drive ----------
TAMBON_SEARCH_DIRS = ["/content/drive/MyDrive/PM25_Saraburi_notebooks/boundary",   # ที่แนะนำ
                      "/content/drive/MyDrive/PM25_Saraburi_notebooks",
                      "/content/drive/MyDrive"]                                    # ค้นหาลึกสุด 4 ชั้น
TAMBON_NAME_PAT = r"(tambon|sub.?district|subdist|adm3|admin3|ตำบล|tb_?|_tam)"
SARA_BBOX = (100.55, 14.25, 101.40, 15.05)
MISSING_PARTS = []
TAMBON_CACHE = "saraburi_tambon_boundary.gpkg"   # สำเนาเฉพาะสระบุรี (เล็ก อ่านเร็ว) สร้างอัตโนมัติข้างไฟล์ต้นฉบับ

def find_tambon_files():
    """คืนรายการ path ของ shapefile/gpkg/geojson/zip ที่อาจเป็นขอบเขตตำบลใน Drive (TAMBON_SHP ก่อน แล้วค้นหาอัตโนมัติ)"""
    import re, glob
    MISSING_PARTS.clear()
    if repo_file(TAMBON_CACHE): return [repo_file(TAMBON_CACHE)]          # repository: data/processed/
    if not os.path.isdir("/content/drive/MyDrive"):
        try:
            from google.colab import drive; drive.mount("/content/drive")
        except Exception as e:
            raise RuntimeError(f"mount Drive ไม่ได้: {str(e)[:80]}")
    p = (globals().get("TAMBON_SHP") or "").strip()
    if p:
        if not p.startswith("/"): p = os.path.join("/content/drive/MyDrive", p)
        if os.path.exists(p): return [p]
        raise RuntimeError(f"ไม่พบไฟล์ TAMBON_SHP = {p}")
    exts = (".shp", ".gpkg", ".geojson", ".json", ".zip"); found = []
    for root0 in TAMBON_SEARCH_DIRS:
        if not os.path.isdir(root0): continue
        hits, base_depth = [], root0.rstrip("/").count("/")
        for root, dirs, files in os.walk(root0):
            if root.count("/") - base_depth >= 4: dirs[:] = []
            dirs[:] = [d for d in dirs if not d.startswith(".") and d not in ("SatPM25_cache", "ERA5_cache", "WorldPop_cache", "GDAS", "PCD_PM25")]
            for fn in files:
                if fn.lower().endswith(".shx") and re.search(TAMBON_NAME_PAT, fn, re.I) and \
                        (fn[:-4] + ".shp").lower() not in {f.lower() for f in files}:
                    MISSING_PARTS.append(os.path.join(root, fn[:-4] + ".shp"))         # มี .shx แต่ไม่มี .shp
                anyname = root.rstrip("/").endswith("/boundary")          # ในโฟลเดอร์ boundary รับทุกชื่อไฟล์
                if fn.lower().endswith(exts) and (anyname or re.search(TAMBON_NAME_PAT, fn, re.I)) \
                        and not re.search(r"(rates|gistar|lisa|roadside|summary|result)", fn, re.I):
                    if fn.lower().endswith(".shp"):
                        stem = fn[:-4]; have = {f.lower() for f in files}
                        if not {stem.lower() + ".dbf", stem.lower() + ".shx"} <= have:
                            MISSING_PARTS.append(os.path.join(root, fn)); continue      # .dbf/.shx ยังไม่ครบ (อัปโหลดไม่เสร็จ?)
                    hits.append(os.path.join(root, fn))
            if root0 == "/content/drive/MyDrive" and len(hits) > 50: break
        if hits:
            order = {".shp": 0, ".gpkg": 1, ".zip": 2, ".geojson": 3, ".json": 4}
            rank = lambda h: 0 if re.search(TAMBON_NAME_PAT, os.path.basename(h), re.I) else \
                             2 if re.search(r"(adm[0-2]|admin[0-2]|amphoe|province|district|_ln|_pt|line|point)", os.path.basename(h), re.I) else 1
            hits.sort(key=lambda h: (rank(h), order[os.path.splitext(h)[1].lower()], len(h)))
            found += [h for h in hits if h not in found]
            if any(os.path.basename(h) == TAMBON_CACHE for h in found): break      # เจอสำเนาแล้ว ไม่ต้องค้นต่อ
    found.sort(key=lambda h: os.path.basename(h) != TAMBON_CACHE)                # สำเนาเฉพาะสระบุรีมาก่อน
    if found: return found
    extra = f" (พบ .shp ที่ไฟล์ประกอบไม่ครบ: {[os.path.basename(m) for m in MISSING_PARTS[:3]]})" if MISSING_PARTS else ""
    raise RuntimeError("ไม่พบ shapefile ตำบลใน Drive" + extra + " — วางไฟล์ไว้ที่ MyDrive/PM25_Saraburi_notebooks/boundary/ "
                       "(ชื่อไฟล์มีคำว่า tambon/subdistrict/ตำบล) หรือกำหนด TAMBON_SHP")

def read_tambon_file(path):
    """อ่านไฟล์ขอบเขต + แก้ภาษาไทยเพี้ยน (TIS-620) + กำหนด CRS + ตัดเฉพาะสระบุรี + หาคอลัมน์รหัส/ชื่อตำบล"""
    import re
    src, layer = path, None
    if path.lower().endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            inner = [n for n in z.namelist() if n.lower().endswith((".shp", ".gpkg", ".geojson", ".json"))]
        if not inner: raise RuntimeError(f"ใน {os.path.basename(path)} ไม่มี .shp/.gpkg/.geojson")
        lvl = lambda n: (0 if re.search(r"(adm3|admin3|tambon|sub.?dist|ตำบล)", n, re.I) else
                         2 if re.search(r"(adm[0-2]|admin[0-2]|amphoe|province|district|line|pt|point)", n, re.I) else 1)
        inner.sort(key=lambda n: (lvl(n), len(n)))
        src = f"zip://{path}!{inner[0]}"
    if src.lower().endswith(".gpkg"):
        try:
            import pyogrio
            names = [l[0] for l in pyogrio.list_layers(src)]
        except Exception:
            names = []
        if len(names) > 1:
            pref = [n for n in names if re.search(r"(adm3|admin3|tambon|sub.?dist|ตำบล)", n, re.I)]
            layer = pref[0] if pref else names[-1]
    kw = {"layer": layer} if layer else {}
    g = None
    if os.path.getsize(path) > 50e6:                    # ไฟล์ทั้งประเทศขนาดใหญ่ → อ่านเฉพาะกรอบสระบุรี (เร็วกว่ามาก)
        try:
            g = gpd.read_file(src, bbox=(SARA_BBOX[0] - .3, SARA_BBOX[1] - .3, SARA_BBOX[2] + .3, SARA_BBOX[3] + .3), **kw)
            if len(g) < MIN_TAMBONS: g = None           # พิกัดในไฟล์ไม่ใช่ lat/lon → อ่านทั้งไฟล์
            else: kw["bbox"] = (SARA_BBOX[0] - .3, SARA_BBOX[1] - .3, SARA_BBOX[2] + .3, SARA_BBOX[3] + .3)
        except Exception:
            g = None
    if g is None: g = gpd.read_file(src, **kw)
    thai = lambda s: bool(re.search(r"[฀-๿]", str(s)))
    garbled = lambda s: bool(re.search(r"[À-ÿ]{2,}", str(s)))
    is_txt = lambda c: c != "geometry" and not pd.api.types.is_numeric_dtype(g[c])
    obj = [c for c in g.columns if is_txt(c)]
    if obj and not any(g[c].astype(str).map(thai).any() for c in obj) and any(g[c].astype(str).map(garbled).any() for c in obj):
        for enc in ("cp874", "tis-620"):
            try:
                g2 = gpd.read_file(src, encoding=enc, **kw)
                if any(g2[c].astype(str).map(thai).any() for c in obj if c in g2): g = g2; obj = [c for c in g.columns if is_txt(c)]; print(f"   อ่านภาษาไทยด้วย encoding={enc}"); break
            except Exception: pass
    g = g[g.geometry.notna() & g.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    if g.empty: raise RuntimeError("ไม่มี polygon (อาจเป็นไฟล์จุด/เส้น)")
    if g.crs is None:
        x0, y0, x1, y1 = g.total_bounds
        if -180 <= x0 <= 180 and -90 <= y0 <= 90: g = g.set_crs("EPSG:4326")
        else: g = g.set_crs(UTM); print("   ⚠ ไฟล์ไม่มี .prj — สมมติเป็น UTM 47N (EPSG:32647)")
    g = g.to_crs("EPSG:4326")
    up = {c: c.upper() for c in g.columns}
    digits = lambda s: s.astype(str).str.replace(r"\.0$", "", regex=True).str.replace(r"\D", "", regex=True)
    # ---- รหัสตำบล 6 หลัก ----
    code = None
    for c in g.columns:
        if c == "geometry": continue
        d = digits(g[c])
        if (d.str.len() == 6).mean() > 0.8 and d.str[:2].nunique() <= 80: code = d.where(d.str.len() == 6); break
    if code is None:
        pc = next((c for c in g.columns if re.fullmatch(r"(P|PV|PROV|PRO|CHANGWAT|ADM1)_?(CODE|ID|CD)", up[c])), None)
        ac = next((c for c in g.columns if re.fullmatch(r"(A|AP|AMP|AMPHOE|DIST|ADM2)_?(CODE|ID|CD)", up[c])), None)
        tc = next((c for c in g.columns if re.fullmatch(r"(T|TB|TAM|TAMBON|SUBDIST|ADM3)_?(CODE|ID|CD)", up[c])), None)
        if pc and ac and tc:
            p, a, t = digits(g[pc]).str[-2:].str.zfill(2), digits(g[ac]).str[-2:].str.zfill(2), digits(g[tc]).str[-2:].str.zfill(2)
            code = p + a + t
    # ---- ตัดเฉพาะจังหวัดสระบุรี ----
    n_all = len(g); keep = None
    if code is not None and (code.str[:2] == "19").sum() >= MIN_TAMBONS: keep = code.str[:2] == "19"
    if keep is None:
        for c in g.columns:
            if is_txt(c):
                k = g[c].astype(str).str.contains(r"สระบุรี|Saraburi", case=False, na=False)
                if MIN_TAMBONS <= k.sum() < len(g): keep = k; break
    if keep is None and n_all > 140:
        rp = g.representative_point(); w, s, e, n = SARA_BBOX
        keep = rp.x.between(w, e) & rp.y.between(s, n)
        print("   ⚠ ไม่พบคอลัมน์จังหวัด — ตัดด้วยกรอบพิกัดโดยประมาณ (ตรวจตำบลริมขอบจังหวัดอีกครั้ง)")
    if keep is not None:
        g = g[keep.values].copy(); code = code[keep.values] if code is not None else None
    # ---- ชื่อตำบล ----
    pick = lambda pats: next((c for p in pats for c in g.columns if re.fullmatch(p, up[c])), None)
    nc = pick([r"TAM_?NAMT", r"T_?NAME_?T", r"TB_?TN", r"TAMBON_?T(H)?", r"ADM3_?TH", r"NAME_?TH", r"TAM_?NAME", r"T_?NAME", r"TAMBON", r"ADM3_?EN", r"NAME_?3", r"NAME"])
    if nc is None:
        nc = next((c for c in g.columns if is_txt(c) and g[c].astype(str).map(thai).mean() > 0.8), None)
    ec = pick([r"TAM_?NAME_?E", r"TAM_?NAMEE", r"T_?NAME_?E", r"TB_?EN", r"ADM3_?EN", r"NAME_?EN", r"NAME_?3"])
    g["name"] = g[nc].astype(str).str.replace(r"^(ตำบล|ต\.)\s*", "", regex=True) if nc else [f"T{i+1:03d}" for i in range(len(g))]
    if ec and ec != nc: g["name_en"] = g[ec].astype(str)
    if code is not None: g["a_code"] = code.values
    g["tambon_id"] = g["a_code"] if "a_code" in g else g["name"]
    cols = ["tambon_id"] + (["a_code"] if "a_code" in g else []) + ["name"] + (["name_en"] if "name_en" in g else []) + ["geometry"]
    info = f"รหัส={'มี' if 'a_code' in g else 'ไม่มี'}, ชื่อ={nc}, ตัดจาก {n_all} → {len(g)} หน่วย"
    return g[cols].reset_index(drop=True), info

def load_tambons_drive():
    import time
    errs = []; t0 = time.time()
    for path in find_tambon_files()[:6]:
        short = path.replace("/content/drive/MyDrive/", "MyDrive/")
        try:
            g, info = read_tambon_file(path)
            if len(g) < MIN_TAMBONS: raise RuntimeError(f"ได้ {len(g)} หน่วย ({info})")
            print(f"   ไฟล์: {short} ({info}; {time.time() - t0:.0f} s)")
            if os.path.basename(path) != TAMBON_CACHE:
                try:                                      # ครั้งต่อไปอ่านสำเนานี้แทน (ไม่กี่วินาที)
                    cp = os.path.join(os.path.dirname(path), TAMBON_CACHE)
                    g.to_file(cp, driver="GPKG"); print(f"   บันทึกสำเนาเฉพาะสระบุรี → {cp.replace('/content/drive/MyDrive/', 'MyDrive/')}")
                except Exception as e:
                    print(f"   (บันทึกสำเนาไม่ได้: {str(e)[:60]})")
            return g, f"Drive: {os.path.basename(path)}"
        except Exception as e:
            errs.append(f"{os.path.basename(path)}: {str(e)[:70]}")
    raise RuntimeError("ไฟล์ใน Drive ใช้ไม่ได้ — " + " | ".join(errs))

MIN_TAMBONS = 60     # สระบุรีมี ~108–111 ตำบล ถ้าได้น้อยกว่านี้แสดงว่าได้ขอบเขตผิดระดับ (เช่น อำเภอ)

def clean_units(g):
    """ตัด polygon ซ้ำ และ polygon 'แม่' ที่ครอบหน่วยอื่น ≥ 2 หน่วย (เช่น อำเภอ/จังหวัดที่ปนมา)"""
    n0 = len(g)
    g = g[g.geometry.notna() & ~g.geometry.is_empty].copy()
    g["geometry"] = g.geometry.buffer(0)
    key = g.to_crs(UTM).geometry.apply(lambda x: (round(x.area / 1e4), round(x.centroid.x / 50), round(x.centroid.y / 50)))
    g = g[~key.duplicated()].reset_index(drop=True)
    rp = g.representative_point(); sidx = rp.sindex
    area = g.to_crs(UTM).area.values
    parent = []
    for i, geom in enumerate(g.geometry):
        inside = [j for j in sidx.query(geom, predicate="contains") if j != i and area[j] < 0.8 * area[i]]
        parent.append(len(inside) >= 2)               # ครอบหน่วยที่เล็กกว่าอย่างน้อย 2 หน่วย = ขอบเขตระดับบน
    g = g[~np.array(parent)].reset_index(drop=True)
    if "name" in g:
        g = g[~g["name"].astype(str).str.match(r"^(อำเภอ|จังหวัด|เขต)")].reset_index(drop=True)
    if len(g) != n0:
        print(f"   ทำความสะอาดขอบเขต: {n0} → {len(g)} หน่วย (ตัด polygon ซ้ำ/ระดับอำเภอ-จังหวัดที่ปนมา)")
    return g

KV_URL = "https://raw.githubusercontent.com/kongvut/thai-province-data/master/api/latest/sub_district.json"

def add_tambon_codes(g):
    """เติมรหัสตำบล 6 หลัก (มท.) ให้ขอบเขตที่ไม่มีรหัส (GADM/OSM): จับคู่ตำแหน่งตำบลก่อน แล้วจับคู่ชื่อ"""
    import re
    js = http_get(KV_URL, timeout=(15, 120)).json()
    kv = pd.DataFrame([{"a_code": str(t["id"]), "th": (t["name"] or {}).get("th", ""), "en": (t["name"] or {}).get("en", ""),
                        "lat": t.get("lat"), "lon": t.get("long")} for t in js if t.get("id")])
    pts = kv.dropna(subset=["lat", "lon"])
    pts = gpd.GeoDataFrame(pts, geometry=gpd.points_from_xy(pts.lon, pts.lat), crs="EPSG:4326")
    j = gpd.sjoin(pts, g[["geometry"]].reset_index().rename(columns={"index": "gi"}), predicate="within")
    prov = j.a_code.str[:2].mode().iloc[0]
    kv = kv[kv.a_code.str[:2] == prov]
    norm_th = lambda x: re.sub(r"^(ตำบล|ต\.)", "", str(x)).replace(" ", "").strip()
    norm_en = lambda x: re.sub(r"[^a-z]", "", str(x).lower())
    by_th = dict(zip(kv.th.map(norm_th), kv.a_code)); by_en = dict(zip(kv.en.map(norm_en), kv.a_code))
    code = pd.Series(index=g.index, dtype=object)
    for col, table, f in [("name", by_th, norm_th), ("name_en", by_en, norm_en), ("name", by_en, norm_en)]:
        if col in g:
            m = g[col].map(f).map(table); code = code.fillna(m)
    jj = j[j.a_code.str[:2] == prov].drop_duplicates("gi").set_index("gi")["a_code"]
    code = code.fillna(jj.reindex(g.index))
    g["a_code"] = code
    th = dict(zip(kv.a_code, kv.th)); g["name_th"] = g["a_code"].map(th)
    if g["name_th"].notna().mean() > 0.8: g["name"] = g["name_th"].fillna(g["name"])
    print(f"   เติมรหัสตำบล (มท.) ได้ {g.a_code.notna().sum()} / {len(g)} หน่วย (จังหวัดรหัส {prov})")
    return g.drop(columns="name_th")

def load_tambons():
    loaders = {"drive": load_tambons_drive, "cod": load_tambons_cod, "gadm": load_tambons_gadm, "osm": load_tambons_osm}
    first = BOUNDARY_SOURCE if BOUNDARY_SOURCE in loaders else "drive"
    order = [loaders[first]] + [f for k, f in loaders.items() if k != first]
    for f in order:
        try:
            g, src = f(); g = clean_units(g.reset_index(drop=True))
            if len(g) < MIN_TAMBONS:
                raise RuntimeError(f"ได้เพียง {len(g)} หน่วย — น่าจะไม่ใช่ระดับตำบล")
            if "a_code" not in g or g["a_code"].isna().mean() > 0.2:
                try:
                    g = add_tambon_codes(g)
                except Exception as e:
                    print(f"   ⚠ เติมรหัสตำบลไม่ได้: {str(e)[:80]} — การจับคู่ข้อมูล HDC จะใช้ชื่อแทน")
            print(f"✓ ขอบเขตตำบล: {len(g)} ตำบล  ({src})"); return g, src
        except Exception as e:
            print(f"   ✗ {f.__name__}: {str(e)[:120]}")
    raise RuntimeError("โหลดขอบเขตตำบลไม่สำเร็จ")

WORLDPOP_URLS = [
    "https://data.worldpop.org/GIS/Population/Global_2000_2020_1km_UNadj/2020/THA/tha_ppp_2020_1km_Aggregated_UNadj.tif",  # ~3 MB
    "https://data.worldpop.org/GIS/Population/Global_2000_2020/2020/THA/tha_ppp_2020_UNadj.tif",                            # ~250 MB (100 m)
]

def add_population(g):
    """ดาวน์โหลด GeoTIFF ของ WorldPop ลงเครื่อง แล้วรวมประชากรรายตำบล (zonal sum)"""
    import rasterio
    from rasterio.mask import mask
    for u in [WORLDPOP_URLS[0]] + WORLDPOP_URLS:          # ไฟล์ 1 km ลอง 2 ครั้ง (server ตัดการเชื่อมต่อบ่อย) แล้วจึงใช้ 100 m
        fn = os.path.join("/content" if os.path.isdir("/content") else ".", u.split("/")[-1])
        try:
            if not (os.path.exists(fn) and os.path.getsize(fn) > 1e6):
                print(f"   ดาวน์โหลด {u.split('/')[-1]} ...")
                with requests.get(u, headers=UA, stream=True, timeout=(15, 600)) as r:
                    r.raise_for_status()
                    with open(fn, "wb") as f:
                        for chunk in r.iter_content(1 << 20): f.write(chunk)
            with rasterio.open(fn) as src:
                gg = g.to_crs(src.crs); pop = []
                for geom in gg.geometry:
                    arr, _ = mask(src, [geom], crop=True, all_touched=False, filled=True, nodata=0)
                    a = arr[0].astype(float); a[~np.isfinite(a) | (a < 0)] = 0; pop.append(a.sum())
            if sum(pop) <= 0:
                raise RuntimeError("ผลรวมประชากร = 0")
            g["pop"] = np.round(pop); print(f"✓ ประชากร WorldPop 2020: รวม {g['pop'].sum():,.0f} คน  ({u.split('/')[-1]})")
            return g, "WorldPop 2020 UN-adjusted (" + u.split("/")[-1] + ")"
        except Exception as e:
            print(f"   ✗ WorldPop {u.split('/')[-1]}: {type(e).__name__}: {str(e)[:100]}")
            if os.path.exists(fn) and os.path.getsize(fn) < 1e6: os.remove(fn)
    print("⚠ ไม่ได้ข้อมูลประชากร — ใช้พื้นที่แทนเป็นตัวหารสำรอง"); g["pop"] = np.nan
    return g, None

FIRMS_PRODUCTS = ["viirs-snpp", "viirs-noaa20", "modis"]       # ลำดับความชอบ (VIIRS 375 m ละเอียดกว่า MODIS 1 km)
FIRMS_API_SRC = {"viirs-snpp": ["VIIRS_SNPP_SP", "VIIRS_SNPP_NRT"], "viirs-noaa20": ["VIIRS_NOAA20_SP", "VIIRS_NOAA20_NRT"],
                 "modis": ["MODIS_SP", "MODIS_NRT"]}

def _firms_key():
    if FIRMS_MAP_KEY: return FIRMS_MAP_KEY
    try:
        from google.colab import userdata; return userdata.get("FIRMS_MAP_KEY")
    except Exception:
        return None

FIRMS_GAPS = []        # ช่วงวันที่ API ไม่มีข้อมูล (ไม่ใช่ "ไม่มีไฟ") — ใช้ตัดออก/ปรับตัวหาร

def firms_api_year(key, sources, y, bbox):
    """FIRMS area API ทีละ 5 วัน: ใช้ SP (standard, ตรวจสอบแล้ว) ก่อน แล้วต่อด้วย NRT เฉพาะวันหลังวันสุดท้ายของ SP
    (SP ออกช้ากว่าจริง ~2–3 เดือน ; NRT เก็บย้อนหลังแค่ ~2 เดือน → ต้องใช้ทั้งสองต่อกันจึงครบปีปัจจุบัน)"""
    end = min(pd.Timestamp(f"{y}-12-31"), pd.Timestamp.now().normalize())
    got, how, last = [], [], None
    for src in sources:
        out, bad = [], 0
        start = pd.Timestamp(f"{y}-01-01") if last is None else last + pd.Timedelta(days=1)
        for d in pd.date_range(start, end, freq="5D"):
            u = (f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{key}/{src}/"
                 f"{bbox[0]:.3f},{bbox[1]:.3f},{bbox[2]:.3f},{bbox[3]:.3f}/5/{d:%Y-%m-%d}")
            try:
                txt = http_get(u, timeout=(15, 120), tries=3).text
                if "latitude" not in txt[:300]:
                    bad += 1
                    if bad == 1: print(f"   FIRMS API {src}: {txt[:120].strip()}")
                    if "invalid" in txt.lower() or "key" in txt.lower(): break
                    continue
                t = pd.read_csv(io.StringIO(txt))
                if len(t): out.append(t[pd.to_datetime(t.acq_date) <= end])
            except Exception as e:
                bad += 1
        if out:
            t = pd.concat(out, ignore_index=True)
            t = t[pd.to_datetime(t.acq_date) >= start]
            if len(t):
                lo, hi = pd.to_datetime(t.acq_date).min(), pd.to_datetime(t.acq_date).max()
                if got and (lo - start).days > 2:
                    print(f"   ⚠ FIRMS {y}: ไม่มีข้อมูลช่วง {start:%Y-%m-%d} → {lo - pd.Timedelta(days=1):%Y-%m-%d} (SP ยังไม่ออก/NRT ลบไปแล้ว)")
                    FIRMS_GAPS.append((start, lo - pd.Timedelta(days=1)))
                got.append(t)
                how.append(f"{src} {lo:%d/%m}–{hi:%d/%m}"); last = hi
        if last is not None and last >= end - pd.Timedelta(days=2): break
    if not got: return None, None
    t = pd.concat(got, ignore_index=True).drop_duplicates()
    print(f"   FIRMS {y} ผ่าน API: " + " + ".join(how) + f" (ถึง {pd.to_datetime(t.acq_date).max():%Y-%m-%d})")
    return t, "API " + "+".join(how)

def _firms_year(product, y, bbox):
    try:
        url = f"https://firms.modaps.eosdis.nasa.gov/data/country/{product}/{y}/{product}_{y}_Thailand.csv"
        return pd.read_csv(io.BytesIO(http_get(url, timeout=(15, 300), tries=2).content)), "yearly"
    except Exception:
        key = _firms_key()
        if not key: return None, None
        t, how = firms_api_year(key, FIRMS_API_SRC[product], y, bbox)
        return (t, how) if t is not None else (None, None)

def load_fires(g):
    """จุดความร้อน VIIRS S-NPP 375 m รายปี (เครื่องวัดเดียวตลอดช่วง เพื่อให้เทียบข้ามปีได้)
    ปีที่ไม่มีไฟล์รายปี (เช่น ปีปัจจุบัน) → ดึงผ่าน API ถ้ามี FIRMS_MAP_KEY มิฉะนั้นตัดปีนั้นออก (ไม่เติมค่า 0)"""
    global YEARS, YEAR_FRAC
    b = g.total_bounds; bbox = (b[0] - 0.05, b[1] - 0.05, b[2] + 0.05, b[3] + 0.05)
    prod, today = "viirs-snpp", pd.Timestamp.now().normalize()
    frames, how, got = [], [], []
    for y in YEARS:
        if y > today.year: continue
        t, h = _firms_year(prod, y, bbox)
        if t is None:
            print(f"   ✗ FIRMS {prod} {y}: ไม่มีไฟล์รายปี" + ("" if _firms_key() else " — ใส่ FIRMS_MAP_KEY (ฟรี) เพื่อดึงปีนี้ผ่าน API"))
            continue
        t = t[pd.to_numeric(t.longitude, errors="coerce").between(bbox[0], bbox[2]) &
              pd.to_numeric(t.latitude, errors="coerce").between(bbox[1], bbox[3])]
        frames.append(t); how.append(f"{y}:{h}"); got.append(y)
        print(f"   ✓ FIRMS {prod} {y} ({h}): {len(t):,} จุดในกรอบจังหวัด", flush=True)
    if not frames:
        raise RuntimeError("ดึง FIRMS ไม่ได้ — ใส่ FIRMS_MAP_KEY (ขอฟรีที่ https://firms.modaps.eosdis.nasa.gov/api/map_key)")
    missing = [y for y in YEARS if y not in got]
    if missing:
        print(f"⚠ ไม่มีข้อมูลจุดความร้อนปี {missing} — ตัดปีนั้นออกจากการวิเคราะห์")
    YEARS = got
    F = pd.concat(frames, ignore_index=True)
    ok = F["confidence"].astype(str).str.lower().str[0].isin(["n", "h"])
    if "type" in F:                                   # type 0 = presumed vegetation fire (ตัดแหล่งความร้อนถาวร เช่น เตาเผาปูน)
        ok &= pd.to_numeric(F["type"], errors="coerce").fillna(0).eq(0)
    F = F[ok].copy()
    F["time_local"] = pd.to_datetime(F["acq_date"].astype(str) + " " + F["acq_time"].astype(int).astype(str).str.zfill(4),
                                     format="%Y-%m-%d %H%M") + pd.Timedelta(hours=7)
    F = F[F.time_local.dt.year.isin(YEARS)]
    pts = gpd.GeoDataFrame(F, geometry=gpd.points_from_xy(F.longitude, F.latitude), crs="EPSG:4326")
    pts = gpd.sjoin(pts, g[["geometry"]], predicate="within", how="inner").rename(columns={"index_right": "tidx"})
    pts["year"] = pts.time_local.dt.year; pts["month"] = pts.time_local.dt.to_period("M")
    YEAR_FRAC = {y: 1.0 for y in YEARS}               # ปีที่ยังไม่ครบ (ปีปัจจุบัน) นับตามสัดส่วนวันที่มีข้อมูล
    if today.year in YEARS:
        gap_days = sum((e - s).days + 1 for s, e in FIRMS_GAPS if s.year == today.year)
        YEAR_FRAC[today.year] = ((pts.time_local.max().normalize() - pd.Timestamp(today.year, 1, 1)).days + 1 - gap_days) / 365.0
    print(f"✓ FIRMS {prod}: {len(pts):,} จุดความร้อนในจังหวัด ปี {YEARS[0]}–{YEARS[-1]}")
    return pts, f"NASA FIRMS {prod} ({', '.join(how)})"

TAMBON, BND_SRC = load_tambons()
TAMBON["area_km2"] = TAMBON.to_crs(UTM).area / 1e6
TAMBON, POP_SRC = add_population(TAMBON)
FIRES, FIRE_SRC = load_fires(TAMBON)
for y in YEARS:
    TAMBON[f"fires_{y}"] = np.bincount(FIRES.loc[FIRES.year == y, "tidx"], minlength=len(TAMBON))
TAMBON["fires_all"] = TAMBON[[f"fires_{y}" for y in YEARS]].sum(axis=1)
TAMBON["density_km2"] = TAMBON.fires_all / TAMBON.area_km2 / sum(YEAR_FRAC.values())    # จุด/km²/ปี
DATA_SOURCES = {"boundaries": BND_SRC, "population": POP_SRC, "fires": FIRE_SRC}
print(pd.Series(DATA_SOURCES))
TAMBON.drop(columns="geometry").describe().round(2)

## 2. ตรวจคุณภาพข้อมูล (QC)
### 2.1 ขอบเขตตำบล
ตรวจจำนวนตำบล, polygon ที่ไม่ถูกต้อง, ช่องว่าง/การทับซ้อนระหว่างตำบล (มีผลต่อ spatial weights ใน LISA)

In [ ]:
qc = {"tambons": len(TAMBON), "expected (HDC)": EXPECTED_TAMBONS,
      "invalid geometries": int((~TAMBON.is_valid).sum()),
      "duplicate names": int(TAMBON.name.duplicated().sum()),
      "area min (km²)": round(TAMBON.area_km2.min(), 2), "area max (km²)": round(TAMBON.area_km2.max(), 2)}
u = TAMBON.to_crs(UTM)
union = u.geometry.union_all() if hasattr(u.geometry, "union_all") else u.unary_union
qc["sum of areas − union area (km², ≈0 = no overlaps)"] = round((u.area.sum() - union.area) / 1e6, 3)
qc["holes inside province (km²)"] = round(sum(sum(__import__("shapely").geometry.Polygon(r).area for r in g.interiors)
                                             for g in getattr(union, "geoms", [union])) / 1e6, 3)
display(pd.Series(qc))
if len(TAMBON) != EXPECTED_TAMBONS:
    print(f"⚠ จำนวนตำบล ({len(TAMBON)}) ≠ {EXPECTED_TAMBONS} — ขอบเขตอาจเป็นคนละรุ่นกับ HDC (ตำบลตั้งใหม่/ยุบรวม) ตรวจก่อนจับคู่ข้อมูล")
if qc["invalid geometries"]:
    TAMBON["geometry"] = TAMBON.buffer(0); print("แก้ invalid geometries ด้วย buffer(0) แล้ว")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 6.5))
TAMBON.plot(ax=axes[0], facecolor="#f4f4f4", edgecolor="k", lw=0.4)
for r in TAMBON.itertuples():
    p = r.geometry.representative_point(); axes[0].text(p.x, p.y, str(r.Index), fontsize=5, ha="center")
axes[0].set_title(f"{len(TAMBON)} tambons (index numbers)", fontsize=9)
if TAMBON["pop"].notna().any():
    TAMBON.assign(dens=TAMBON["pop"] / TAMBON.area_km2).plot("dens", ax=axes[1], scheme="quantiles", k=5, cmap="Blues",
        edgecolor="grey", lw=0.3, legend=True, legend_kwds={"fontsize": 7, "loc": "lower left", "title": "persons/km²"})
axes[1].set_title(f"Population density ({DATA_SOURCES['population']})", fontsize=9)
TAMBON.plot("density_km2", ax=axes[2], scheme="quantiles", k=5, cmap="YlOrRd", edgecolor="grey", lw=0.3,
            legend=True, legend_kwds={"fontsize": 7, "loc": "lower left", "title": "fires/km²/yr"})
FIRES.sample(min(3000, len(FIRES)), random_state=0).plot(ax=axes[2], color="k", markersize=0.5, alpha=0.4)
axes[2].set_title("FIRMS fire detections", fontsize=9)
for a in axes: a.set_axis_off()
plt.tight_layout(); plt.show()

### 2.2 จุดความร้อน: แนวโน้มรายเดือน และการกระจายรายตำบล

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 3.8))
FIRES.groupby("month").size().reindex(pd.period_range(FIRES["month"].min(), FIRES["month"].max(), freq="M"), fill_value=0
                                      ).plot(kind="bar", ax=axes[0], color="#e67e22", width=0.9)
axes[0].set_title("detections per month", fontsize=9); axes[0].tick_params(axis="x", labelsize=6)
axes[1].hist(TAMBON.fires_all, bins=30, color="grey"); axes[1].set_title("fires per tambon (all years)", fontsize=9)
yr = TAMBON[[f"fires_{y}" for y in YEARS]]
axes[2].imshow(np.corrcoef(yr.T.values), vmin=0, vmax=1, cmap="viridis")
axes[2].set_xticks(range(len(YEARS)), YEARS); axes[2].set_yticks(range(len(YEARS)), YEARS)
for i in range(len(YEARS)):
    for j in range(len(YEARS)):
        axes[2].text(j, i, f"{np.corrcoef(yr.T.values)[i, j]:.2f}", ha="center", color="w")
axes[2].set_title("between-year correlation of tambon counts", fontsize=9)
plt.tight_layout(); plt.show()
print(f"ตำบลที่ไม่มีจุดความร้อนเลย: {(TAMBON.fires_all == 0).sum()} / {len(TAMBON)}")
print(f"ความแปรปรวน/ค่าเฉลี่ย (dispersion) = {TAMBON.fires_all.var() / TAMBON.fires_all.mean():.1f} "
      "(>> 1 = overdispersion → Poisson model อาจต้องปรับ หรือพิจารณา negative binomial ในการวิเคราะห์ต่อ)")

## 3. (ทางเลือก) จับคู่ข้อมูลผู้ป่วย HDC
ส่วนนี้ **ต้องอ่านไฟล์ HDC จาก Drive** (Colab เชื่อมต่อ API ของ HDC ไม่ได้) — ตั้ง `USE_HDC = True` แล้วแก้ชื่อคอลัมน์ให้ตรงกับไฟล์จริง
ผลลัพธ์จะรายงานตำบลที่จับคู่ชื่อไม่ได้ เพื่อแก้ไขก่อนวิเคราะห์ (แนะนำ `BOUNDARY_SOURCE = "osm"` เพราะมีชื่อตำบลภาษาไทย)

In [ ]:
# =====================================================================
#  (ทางเลือก) ใช้ข้อมูลผู้ป่วย HDC ของคุณแทนจุดความร้อน
#  HDC เชื่อมต่อจาก Colab ไม่ได้ จึงต้องอ่านไฟล์ที่ดาวน์โหลดไว้แล้วจาก Drive — ปิดไว้เป็นค่าเริ่มต้น
#  ต้องมีคอลัมน์ชื่อตำบลที่จับคู่กับคอลัมน์ name ของขอบเขตได้ (แนะนำ BOUNDARY_SOURCE="osm" เพราะมีชื่อภาษาไทย)
# =====================================================================
USE_HDC = False
HDC_CSV = repo_file("hdc_saraburi_pm25_disease_tambon_annual_2565_2570.csv") or "/content/drive/MyDrive/000_research/11HealthImpactAssessment_SW/0data/hdc_saraburi_pm25_disease_tambon_FINAL.csv"
HDC_CODE_COL = "a_code"            # รหัสตำบล 6 หลัก (เช่น 190101) — จับคู่แม่นกว่าชื่อ
HDC_TAMBON_COL = "a_name"          # ใช้เมื่อขอบเขตไม่มีรหัสตำบล
HDC_COUNT_COL = "total"            # หรือชื่อโรค เช่น "COPD", "Pneumonia"
HDC_YEAR_COL, HDC_YEAR = "year", None   # เช่น HDC_YEAR = 2568 ; None = รวมทุกปี
if USE_HDC:
    from google.colab import drive; drive.mount("/content/drive")
    h = pd.read_csv(HDC_CSV, encoding="utf-8-sig")
    if HDC_YEAR: h = h[h[HDC_YEAR_COL] == HDC_YEAR]
    if "a_code" in TAMBON and HDC_CODE_COL in h:
        cases = h.groupby(h[HDC_CODE_COL].astype(str).str[:6])[HDC_COUNT_COL].sum()
        TAMBON["cases"] = TAMBON["a_code"].astype(str).map(cases); how = "รหัสตำบล"
    else:
        clean = lambda s: str(s).replace("ตำบล", "").replace("ต.", "").strip()
        cases = h.groupby(h[HDC_TAMBON_COL].map(clean))[HDC_COUNT_COL].sum()
        TAMBON["cases"] = TAMBON["name"].map(clean).map(cases); how = "ชื่อตำบล"
    print(f"จับคู่ด้วย{how}ได้ {TAMBON.cases.notna().sum()} / {len(TAMBON)} ตำบล — ที่จับคู่ไม่ได้:",
          TAMBON.loc[TAMBON.cases.isna(), "name"].tolist()[:15])

In [ ]:
if USE_HDC and TAMBON["pop"].isna().all():
    print("⚠ ไม่มีข้อมูลประชากร (WorldPop ดาวน์โหลดไม่ได้) — ข้ามการคำนวณอัตราต่อประชากร")
elif USE_HDC:
    TAMBON["rate_per_1000"] = TAMBON.cases / TAMBON["pop"] * 1000
    display(TAMBON[["name", "cases", "pop", "rate_per_1000"]].describe().round(2))
    TAMBON.plot("rate_per_1000", scheme="quantiles", k=5, cmap="Reds", edgecolor="grey", lw=0.3, legend=True,
                missing_kwds={"color": "lightgrey", "hatch": "//", "label": "unmatched"}, figsize=(6, 6))
    plt.title("HDC cases per 1,000 population (WorldPop)"); plt.axis("off"); plt.show()

## 4. บันทึกชุดข้อมูลกลาง

In [ ]:
TAMBON.to_file(f"{OUTDIR}/A_tambon_dataset.gpkg", layer="tambon", driver="GPKG")
TAMBON.drop(columns="geometry").to_csv(f"{OUTDIR}/A_tambon_dataset.csv", index=False)
FIRES[["tidx", "latitude", "longitude", "time_local", "year", "confidence"]].assign(month=FIRES.month.astype(str)
      ).to_csv(f"{OUTDIR}/A_fire_points.csv", index=False)
json.dump({**DATA_SOURCES, "qc": {k: (float(v) if isinstance(v, (int, float, np.number)) else v) for k, v in qc.items()}},
          open(f"{OUTDIR}/A_sources_qc.json", "w"), ensure_ascii=False, indent=1)
print("บันทึกแล้ว:", sorted(f for f in os.listdir(OUTDIR) if f.startswith("A_")))

## 5. ขั้นต่อไป
- **Notebook 15 (LISA):** Global Moran's I + LISA + Gi* จากชุดข้อมูลเดียวกัน
- **Notebook 16 (SaTScan):** spatial Poisson scan + space-time permutation scan + ไฟล์สำหรับโปรแกรม SaTScan

**ข้อควรระวังที่พบจาก QC:** จำนวนตำบลไม่ตรงกับ HDC, ตำบลที่มีพื้นที่เล็กมาก (อัตราไม่เสถียร → ใช้ EB ใน B), overdispersion ของจำนวนนับ, และความสอดคล้องระหว่างปีของรูปแบบเชิงพื้นที่

**อ้างอิงข้อมูล:** OCHA COD-AB Thailand (RTSD, HDX cod-ab-tha) · GADM 4.1 · OpenStreetMap contributors (ODbL) · WorldPop (2018) doi:10.5258/SOTON/WP00645 · NASA FIRMS VIIRS 375 m (VNP14IMG)